# RMSIN: Rotated Multi-Scale Interaction Network for RRSIS## Referring Remote Sensing Image Segmentation on RRSIS-D**Text Encoder**: HuggingFace `bert-base-uncased`**Vision Backbone**: MultiModalConvNeXtV2 (ConvNeXt-V2-Base, ImageNet-22K pretrained)**Fusion**: Configurable — PWAM / GCN / CrossModalGNN / GraphSAGE (via `cfg.fusion_type`)**Decoder**: SimpleDecoding with ARC (Adaptive Rotated Convolution) + CIM### Stage-Based TrainingThis notebook supports **incremental training** via boolean toggles:- **Stage 0**: Baseline (no augmentation, no open-set, no KD)- **Stage 1**: + Data Augmentation- **Stage 2**: + Open-Set Network- **Stage 3**: + Knowledge Distillation (teacher = Stage 2 best model)Each stage produces its own checkpoint and results CSV for comparison.

---## 1. Setup & Installation

In [ ]:
!pip install -q einops torchinfo tabulate transformers timm

In [ ]:
# ============================================================# 1a. Environment detection & path setup# ============================================================import sys, os# Detect ColabIS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')if IS_COLAB:    from google.colab import drive    drive.mount('/content/drive', force_remount=False)    # ---- Dataset / Output paths (edit if your Drive layout differs) ----    DATASET_ROOT    = '/content/drive/MyDrive/CSE465/RRSIS-D'    OUTPUT_DIR      = '/content/drive/MyDrive/CSE465/checkpoints'    # ---- Locate the RMSIN repo on Drive (auto-search common locations) ----    # Add your own path first if you know where it is; auto-search is a fallback.    CANDIDATE_REPO_PATHS = [        '/content/drive/MyDrive/CSE465/rmsin_src',        '/content/drive/MyDrive/RRSIS-D_CSE465/Notebooks/rmsin_src',        '/content/drive/MyDrive/rmsin_src',        '/content/rmsin_src',  # in case user manually uploaded it to /content    ]    def _looks_like_rmsin_repo(path):        """A valid repo must contain both lib/ and arc/ folders."""        return (os.path.isdir(path)                and os.path.isdir(os.path.join(path, 'lib'))                and os.path.isdir(os.path.join(path, 'arc')))    RMSIN_REPO_PATH = None    for p in CANDIDATE_REPO_PATHS:        if _looks_like_rmsin_repo(p):            RMSIN_REPO_PATH = p            break    if RMSIN_REPO_PATH is None:        # Search one level under Drive root as a last resort        drive_root = '/content/drive/MyDrive'        if os.path.exists(drive_root):            for d, subdirs, _ in os.walk(drive_root):                # skip deep traversal — only look 3 levels deep                depth = d.replace(drive_root, '').count(os.sep)                if depth > 3:                    subdirs.clear()                    continue                if _looks_like_rmsin_repo(d):                    RMSIN_REPO_PATH = d                    break    if RMSIN_REPO_PATH is None:        raise FileNotFoundError(            "Could not find the RMSIN repo (folder containing both 'lib/' and "            "'arc/' subfolders) on your Google Drive.\n"            "Checked: " + ', '.join(CANDIDATE_REPO_PATHS) + "\n"            "Please upload the rmsin_src folder to Drive, then either:\n"            "  (a) place it at one of the candidate paths above, or\n"            "  (b) add the actual path to CANDIDATE_REPO_PATHS at the top of this cell."        )else:    # Local fallback    RMSIN_REPO_PATH = './RMSIN/rmsin_src'    DATASET_ROOT    = './RRSIS-D'    OUTPUT_DIR      = './checkpoints'# ---- Ensure output dir exists ----os.makedirs(OUTPUT_DIR, exist_ok=True)# ---- Add repo to path (remove any stale entries first) ----# Clean up any old /content/rmsin_src entries from previous sessionssys.path[:] = [p for p in sys.path if p != '/content/rmsin_src' or p == RMSIN_REPO_PATH]if RMSIN_REPO_PATH not in sys.path:    sys.path.insert(0, RMSIN_REPO_PATH)# ---- Dataset subfolders ----IMAGE_DIR  = os.path.join(DATASET_ROOT, 'JPEGImages')LABEL_DIR  = os.path.join(DATASET_ROOT, 'ann_split')# ---- Debug prints ----print("\nSetup Complete:")print("Repo Path :", RMSIN_REPO_PATH)print("Dataset   :", DATASET_ROOT)print("Output    :", OUTPUT_DIR)print("\nVerification:")print("Repo exists?      ", os.path.isdir(RMSIN_REPO_PATH))print("Repo/lib exists?  ", os.path.isdir(os.path.join(RMSIN_REPO_PATH, 'lib')))print("Repo/arc exists?  ", os.path.isdir(os.path.join(RMSIN_REPO_PATH, 'arc')))print("Image dir exists? ", os.path.exists(IMAGE_DIR))print("Label dir exists? ", os.path.exists(LABEL_DIR))if os.path.exists(DATASET_ROOT):    print("\nDataset contents:")    for item in sorted(os.listdir(DATASET_ROOT)):        full = os.path.join(DATASET_ROOT, item)        if os.path.isdir(full):            print(f'  DIR  {item}: {len(os.listdir(full))} items')        else:            print(f'  FILE {item}')

In [ ]:
# ============================================================# 1b. Core Imports# ============================================================import torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.utils.checkpoint as checkpoint_utilfrom torch.utils.data import Dataset, DataLoaderimport numpy as npimport randomimport jsonimport pickleimport timeimport datetimeimport gcimport mathimport cv2import shutilimport subprocessimport matplotlib.pyplot as pltfrom pathlib import Pathfrom collections import OrderedDict, defaultdict, Counterfrom tqdm.auto import tqdmfrom tabulate import tabulatefrom transformers import BertTokenizer, BertModelfrom transformers import ConvNextV2Config, ConvNextV2Modelimport timmfrom timm.models.layers import DropPath, to_2tuple, trunc_normal_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {DEVICE}')if DEVICE.type == 'cuda':    print(f'GPU: {torch.cuda.get_device_name(0)}')    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ============================================================# 1c. Reproducibility# ============================================================def seed_everything(seed=2401):    random.seed(seed)    os.environ['PYTHONHASHSEED'] = str(seed)    np.random.seed(seed)    torch.manual_seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)        torch.backends.cudnn.deterministic = True        torch.backends.cudnn.benchmark = True  # faster convseed_everything(2401)print('Seed set: 2401')

In [ ]:
# ============================================================# 1d. Verify RMSIN repo imports# ============================================================repo_ok = Truetry:    from arc import AdaptiveRotatedConv2d, RountingFunction    print('[OK] arc (AdaptiveRotatedConv2d, RountingFunction)')except ImportError as e:    print(f'[WARN] arc import failed: {e}')    repo_ok = Falsetry:    from lib.mask_predictor import SimpleDecoding as _SD    print('[OK] lib.mask_predictor (SimpleDecoding)')except ImportError as e:    print(f'[WARN] lib.mask_predictor failed: {e}')try:    from lib.various_receptive_utils import VariousReceptive, CIM    print('[OK] lib.various_receptive_utils (VariousReceptive, CIM)')except ImportError as e:    print(f'[WARN] VariousReceptive/CIM import failed: {e}')    repo_ok = Falseif not repo_ok:    print('\n⚠️  Some RMSIN repo imports failed.')    print(f'   Check that {RMSIN_REPO_PATH} contains arc/ and lib/ folders.')else:    print('\n✅ All RMSIN repo imports successful.')

---## 2. Configuration

In [ ]:
# ============================================================# 2. Configuration with Stage Toggles# ============================================================class Config:    """Central config with stage-based training toggles."""    # Paths    dataset_root = DATASET_ROOT    output_dir   = OUTPUT_DIR    resume       = ''    # Dataset    dataset  = 'rrsisd'    splitBy  = 'unc'    img_size = 480    # Model    backbone         = 'convnextv2'                         # 'convnextv2' only (Swin commented out)    convnextv2_model = 'facebook/convnextv2-base-22k-224'    # ── GNN Fusion Type ──    # Options: 'pwam', 'gcn', 'cross_modal_gnn', 'graphsage'    fusion_type = 'graphsage'    # Mixed precision    use_bf16 = True    # Regularization    layer_decay   = 0.7    drop_path     = 0.2    frozen_stages = 1    fusion_drop   = 0.0    # Early stopping    patience = 5    # Training    epochs       = 40    batch_size   = 32    lr           = 3e-5    weight_decay = 1e-2    amsgrad      = False    loss_weight  = 0.1    max_tokens   = 20    # DataLoader    num_workers = 4    pin_memory  = True    # Logging    print_freq = 10    model_id   = 'RMSIN_RRSISD'    # ══════════════════════════════════════════════════════════    # ★★★ STAGE TOGGLES — Enable one at a time, in order ★★★    # ══════════════════════════════════════════════════════════    #    # Stage 0: All False  → baseline    # Stage 1: USE_AUGMENTATION = True  → + data augmentation    # Stage 2: USE_AUGMENTATION = True, USE_OPEN_SET = True  → + open-set    # Stage 3: All True → + knowledge distillation    #    USE_AUGMENTATION = False    USE_OPEN_SET     = False    USE_KD           = False    # Open-set config    open_set_mismatch_prob = 0.15  # % of training samples with mismatched text    # KD config    kd_temperature = 4.0    kd_alpha       = 0.5   # weight for distillation loss vs task loss    @property    def stage_name(self):        if self.USE_KD:            return 'stage3_aug_openset_kd'        elif self.USE_OPEN_SET:            return 'stage2_aug_openset'        elif self.USE_AUGMENTATION:            return 'stage1_aug'        else:            return 'stage0_baseline'    # Swin-related properties (kept for compatibility, not used)    swin_type = 'base'    window12  = True    mha       = ''    @property    def embed_dim(self):        return {'tiny': 96, 'small': 96, 'base': 128, 'large': 192}[self.swin_type]    @property    def depths(self):        return {'tiny': [2,2,6,2], 'small': [2,2,18,2],                'base': [2,2,18,2], 'large': [2,2,18,2]}[self.swin_type]    @property    def num_heads_swin(self):        return {'tiny': [3,6,12,24], 'small': [3,6,12,24],                'base': [4,8,16,32], 'large': [6,12,24,48]}[self.swin_type]    @property    def window_size(self):        return 12 if self.window12 else 7    @property    def mha_list(self):        return [int(x) for x in self.mha.split('-')] if self.mha else [1,1,1,1]cfg = Config()print(f'Backbone: {cfg.backbone}')print(f'Fusion: {cfg.fusion_type}')print(f'Stage: {cfg.stage_name}')print(f'  USE_AUGMENTATION = {cfg.USE_AUGMENTATION}')print(f'  USE_OPEN_SET     = {cfg.USE_OPEN_SET}')print(f'  USE_KD           = {cfg.USE_KD}')print(f'Training: epochs={cfg.epochs} | bs={cfg.batch_size} | lr={cfg.lr}')

---## 3. Data Exploration & Visualization

In [ ]:
# ============================================================# 3a. Load RRSIS-D metadata# ============================================================instances_path = os.path.join(cfg.dataset_root, 'instances.json')with open(instances_path, 'r') as f:    instances = json.load(f)print(f'Images:      {len(instances["images"])}')print(f'Annotations: {len(instances["annotations"])}')print(f'Categories:  {len(instances["categories"])}')# Load referring expressionsrefs_path = os.path.join(cfg.dataset_root, f'refs({cfg.splitBy}).p')with open(refs_path, 'rb') as f:    all_refs = pickle.load(f)print(f'Referring expressions: {len(all_refs)}')split_counts = defaultdict(int)for ref in all_refs:    split_counts[ref['split']] += 1print('Splits:', dict(sorted(split_counts.items())))

In [ ]:
# ============================================================# 3b. Mask decoding utility# ============================================================img_dict = {img['id']: img for img in instances['images']}ann_dict = {ann['id']: ann for ann in instances['annotations']}def decode_mask(ann, img_info):    """Decode COCO-format segmentation into binary mask."""    seg = ann['segmentation']    h, w = img_info['height'], img_info['width']    def _safe_decode(rle, h, w):        """Decode RLE and force to (h, w) if annotation size mismatches."""        m = maskUtils.decode(rle)        if m.shape[:2] != (h, w):            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)        return m    if isinstance(seg, dict):        if isinstance(seg['counts'], list):            rle = maskUtils.frPyObjects(seg, h, w)        else:            rle = seg        m = _safe_decode(rle, h, w)    elif isinstance(seg, list):        if len(seg) > 0 and isinstance(seg[0], dict):            # List of RLE dicts — decode each and accumulate            m = np.zeros((h, w), dtype=np.uint8)            for s in seg:                m += _safe_decode(s, h, w)          # ← was the crash point            m = (m > 0).astype(np.uint8)        elif len(seg) > 0 and isinstance(seg[0], str):            rle = {'counts': seg[0], 'size': [h, w]}            m = _safe_decode(rle, h, w)        else:            # Polygon format            rle = maskUtils.frPyObjects(seg, h, w)            m = maskUtils.decode(rle)            if m.ndim == 3:                m = m.sum(axis=2)            m = (m > 0).astype(np.uint8)    else:        m = np.zeros((h, w), dtype=np.uint8)    if m.ndim == 3:        m = m.sum(axis=2)    return (m > 0).astype(np.uint8)

In [ ]:
# ============================================================# 3c. Visualize samples# ============================================================img_dir = os.path.join(cfg.dataset_root, 'JPEGImages')sample_refs = random.sample(all_refs, min(6, len(all_refs)))fig, axes = plt.subplots(3, 2, figsize=(14, 18))for i, ref in enumerate(sample_refs[:3]):    img_info = img_dict[ref['image_id']]    ann = ann_dict[ref['ann_id']]    img = cv2.imread(os.path.join(img_dir, img_info['file_name']))    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    mask_arr = decode_mask(ann, img_info)    # Image + overlay    overlay = img.copy()    overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 +                             np.array([255, 50, 50]) * 0.5).astype(np.uint8)    sents = [s['sent'] for s in ref['sentences']]    axes[i, 0].imshow(overlay)    axes[i, 0].set_title('\n'.join(sents[:2]), fontsize=9, wrap=True)    axes[i, 0].axis('off')    # Mask    axes[i, 1].imshow(mask_arr, cmap='gray')    axes[i, 1].set_title('Ground Truth Mask')    axes[i, 1].axis('off')plt.tight_layout()plt.show()

---## 4. Dataset & Transforms

In [ ]:
# ============================================================# 4a. Image Transforms — with toggleable augmentation# ============================================================class Compose:    def __init__(self, transforms):        self.transforms = transforms    def __call__(self, image, target):        for t in self.transforms:            image, target = t(image, target)        return image, targetclass Resize:    def __init__(self, h, w):        self.h, self.w = h, w    def __call__(self, image, target):        image = cv2.resize(image, (self.w, self.h), interpolation=cv2.INTER_LINEAR)        target = cv2.resize(target, (self.w, self.h), interpolation=cv2.INTER_NEAREST)        return image, targetclass RandomHorizontalFlip:    def __init__(self, p=0.5):        self.p = p    def __call__(self, image, target):        if random.random() < self.p:            image = np.fliplr(image).copy()            target = np.fliplr(target).copy()        return image, targetclass RandomScaleCrop:    """Random scale (0.8-1.2) then crop back to original size."""    def __init__(self, scale_range=(0.8, 1.2), p=0.5):        self.scale_range = scale_range        self.p = p    def __call__(self, image, target):        if random.random() > self.p:            return image, target        h, w = image.shape[:2]        scale = random.uniform(*self.scale_range)        new_h, new_w = int(h * scale), int(w * scale)        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)        target = cv2.resize(target, (new_w, new_h), interpolation=cv2.INTER_NEAREST)        # Crop or pad back to original size        if new_h > h:            y0 = random.randint(0, new_h - h)            x0 = random.randint(0, new_w - w)            image = image[y0:y0+h, x0:x0+w]            target = target[y0:y0+h, x0:x0+w]        elif new_h < h:            pad_h = h - new_h            pad_w = w - new_w            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)            target = cv2.copyMakeBorder(target, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)        return image, targetclass RandomRotation:    def __init__(self, max_angle=15, p=0.5):        self.max_angle = max_angle        self.p = p    def __call__(self, image, target):        if random.random() > self.p:            return image, target        h, w = image.shape[:2]        angle = random.uniform(-self.max_angle, self.max_angle)        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR,                               borderMode=cv2.BORDER_REFLECT)        target = cv2.warpAffine(target, M, (w, h), flags=cv2.INTER_NEAREST,                                borderMode=cv2.BORDER_REFLECT)        return image, targetclass ColorJitter:    def __init__(self, brightness=0.2, contrast=0.2, saturation=0.2, p=0.5):        self.brightness = brightness        self.contrast = contrast        self.saturation = saturation        self.p = p    def __call__(self, image, target):        if random.random() > self.p:            return image, target        # Brightness        image = image.astype(np.float32)        image *= 1 + random.uniform(-self.brightness, self.brightness)        image = np.clip(image, 0, 255).astype(np.uint8)        # Contrast        mean = image.mean()        image = image.astype(np.float32)        image = (image - mean) * (1 + random.uniform(-self.contrast, self.contrast)) + mean        image = np.clip(image, 0, 255).astype(np.uint8)        return image, targetclass ToTensorPair:    def __call__(self, image, target):        image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0        target = torch.from_numpy(target).long()        return image, targetclass NormalizePair:    def __init__(self, mean, std):        self.mean = torch.tensor(mean).view(3,1,1)        self.std  = torch.tensor(std).view(3,1,1)    def __call__(self, image, target):        return (image - self.mean) / self.std, targetdef get_transform(split='train', img_size=480, use_augmentation=False):    t = [Resize(img_size, img_size)]    if split == 'train':        t.append(RandomHorizontalFlip(0.5))        if use_augmentation:            t.append(RandomScaleCrop(scale_range=(0.8, 1.2), p=0.5))            t.append(RandomRotation(max_angle=15, p=0.5))            t.append(ColorJitter(0.2, 0.2, 0.2, p=0.5))    t += [ToTensorPair(), NormalizePair([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]    return Compose(t)aug_status = 'ENABLED' if cfg.USE_AUGMENTATION else 'DISABLED'print(f'Transforms defined. Augmentation: {aug_status}')

In [ ]:
# ============================================================# 4b. RRSISDDataset — with HuggingFace BERT tokenizer# ============================================================class RRSISDDataset(Dataset):    """    RRSIS-D dataset for referring image segmentation.    Each sample = (image, mask, token_ids, attention_mask).    Uses HF BERT tokenizer (replaces original custom tokenizer).    FIXED: Pre-tokenizes ALL sentences once in __init__ instead of           calling tokenizer per-sample in __getitem__.    """    def __init__(self, cfg, split='train', transform=None):        self.cfg = cfg        self.split = split        self.transform = transform        self.max_tokens = cfg.max_tokens        self.img_dir = os.path.join(cfg.dataset_root, 'JPEGImages')        tokenizer = BertTokenizer.from_pretrained(cfg.bert_type)        # Load metadata        with open(os.path.join(cfg.dataset_root, 'instances.json')) as f:            inst = json.load(f)        self.img_dict = {img['id']: img for img in inst['images']}        self.ann_dict = {ann['id']: ann for ann in inst['annotations']}        with open(os.path.join(cfg.dataset_root, f'refs({cfg.splitBy}).p'), 'rb') as f:            refs = pickle.load(f)        # Filter by split, flatten (ref, sentence) pairs        self.samples = []        for ref in refs:            if ref['split'] == split:                for sent in ref['sentences']:                    self.samples.append({                        'sent': sent['sent'],                        'ann_id': ref['ann_id'],                        'image_id': ref['image_id'],                    })        # ── PRE-TOKENIZE all sentences ONCE ────────────────────        all_sents = [s['sent'] for s in self.samples]        enc = tokenizer(            all_sents,            max_length=self.max_tokens,            padding='max_length',            truncation=True,            return_tensors='pt',        )        self.all_input_ids  = enc['input_ids']       # (N, max_tokens)        self.all_attn_masks = enc['attention_mask']   # (N, max_tokens)        del tokenizer  # free memory — no longer needed        # ────────────────────────────────────────────────────────        print(f'[{split}] {len(self.samples)} samples (pre-tokenized)')    def __len__(self):        return len(self.samples)    def __getitem__(self, idx):        s = self.samples[idx]        img_info = self.img_dict[s['image_id']]        img = cv2.imread(os.path.join(self.img_dir, img_info['file_name']))        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        mask = decode_mask(self.ann_dict[s['ann_id']], img_info)        if self.transform:          img, mask = self.transform(img, mask)          mask = mask.long()        return img, mask, self.all_input_ids[idx], self.all_attn_masks[idx]

In [ ]:
# ============================================================# 4c. Build DataLoaders (augmentation toggle)# ============================================================train_ds = RRSISDDataset(cfg, 'train', get_transform('train', cfg.img_size,                                                       use_augmentation=cfg.USE_AUGMENTATION))val_ds   = RRSISDDataset(cfg, 'val',   get_transform('val',   cfg.img_size))test_ds  = RRSISDDataset(cfg, 'test',  get_transform('val',   cfg.img_size))train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory, drop_last=True)val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False,                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)print(f'Train: {len(train_loader)} batches (bs={cfg.batch_size})')print(f'Val:   {len(val_loader)} batches (bs=4)')print(f'Test:  {len(test_loader)} batches (bs=4)')print(f'Augmentation: {"ON" if cfg.USE_AUGMENTATION else "OFF"}')

---## 5. Vision Backbone### Architecture:```Image → ConvNeXt-V2 Stem → [Stage0 → Stage1 → Stage2 → Stage3] → CIM → multi-scale features                            ↑ each stage: blocks → Fusion(PWAM/GNN) → Gating```

In [ ]:
# ============================================================# 5a. Swin Transformer building blocks [COMMENTED OUT — not used with ConvNeXtV2]# ============================================================# The following Swin blocks are commented out because this notebook# uses ConvNeXtV2 as the backbone. Uncomment if switching to Swin.# # ============================================================# # 5a. Swin Transformer building blocks# # ============================================================# class Mlp(nn.Module):#     """MLP as used in Swin Transformer."""#     def __init__(self, in_features, hidden_features=None, out_features=None,#                  act_layer=nn.GELU, drop=0.):#         super().__init__()#         out_features = out_features or in_features#         hidden_features = hidden_features or in_features#         self.fc1 = nn.Linear(in_features, hidden_features)#         self.act = act_layer()#         self.fc2 = nn.Linear(hidden_features, out_features)#         self.drop = nn.Dropout(drop)#     def forward(self, x):#         x = self.fc1(x)#         x = self.act(x)#         x = self.drop(x)#         x = self.fc2(x)#         x = self.drop(x)#         return x# def window_partition(x, window_size):#     B, H, W, C = x.shape#     x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)#     return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)# def window_reverse(windows, window_size, H, W):#     B = int(windows.shape[0] / (H * W / window_size / window_size))#     x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)#     return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)# class WindowAttention(nn.Module):#     """Window-based multi-head self attention with relative position bias."""#     def __init__(self, dim, window_size, num_heads, qkv_bias=True,#                  qk_scale=None, attn_drop=0., proj_drop=0.):#         super().__init__()#         self.dim = dim#         self.window_size = window_size#         self.num_heads = num_heads#         head_dim = dim // num_heads#         self.scale = qk_scale or head_dim ** -0.5#         self.relative_position_bias_table = nn.Parameter(#             torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))#         coords_h = torch.arange(self.window_size[0])#         coords_w = torch.arange(self.window_size[1])#         coords = torch.stack(torch.meshgrid([coords_h, coords_w]))#         coords_flatten = torch.flatten(coords, 1)#         relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]#         relative_coords = relative_coords.permute(1, 2, 0).contiguous()#         relative_coords[:, :, 0] += self.window_size[0] - 1#         relative_coords[:, :, 1] += self.window_size[1] - 1#         relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1#         self.register_buffer('relative_position_index', relative_coords.sum(-1))#         self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)#         self.attn_drop = nn.Dropout(attn_drop)#         self.proj = nn.Linear(dim, dim)#         self.proj_drop = nn.Dropout(proj_drop)#         trunc_normal_(self.relative_position_bias_table, std=.02)#         self.softmax = nn.Softmax(dim=-1)#     def forward(self, x, mask=None):#         B_, N, C = x.shape#         qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)#         q, k, v = qkv[0], qkv[1], qkv[2]#         q = q * self.scale#         attn = q @ k.transpose(-2, -1)#         rpb = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(#             self.window_size[0] * self.window_size[1],#             self.window_size[0] * self.window_size[1], -1)#         attn = attn + rpb.permute(2, 0, 1).unsqueeze(0)#         if mask is not None:#             nW = mask.shape[0]#             attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)#             attn = attn.view(-1, self.num_heads, N, N)#         attn = self.softmax(attn)#         attn = self.attn_drop(attn)#         x = (attn @ v).transpose(1, 2).reshape(B_, N, C)#         x = self.proj(x)#         x = self.proj_drop(x)#         return x# class SwinTransformerBlock(nn.Module):#     """Swin block with window attention + shifted window attention."""#     def __init__(self, dim, num_heads, window_size=7, shift_size=0,#                  mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0.,#                  attn_drop=0., drop_path=0., act_layer=nn.GELU,#                  norm_layer=nn.LayerNorm):#         super().__init__()#         self.dim = dim#         self.num_heads = num_heads#         self.window_size = window_size#         self.shift_size = shift_size#         self.mlp_ratio = mlp_ratio#         self.norm1 = norm_layer(dim)#         self.attn = WindowAttention(#             dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,#             qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)#         self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()#         self.norm2 = norm_layer(dim)#         self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio),#                        act_layer=act_layer, drop=drop)#         self.H = None#         self.W = None#     def forward(self, x, mask_matrix):#         B, L, C = x.shape#         H, W = self.H, self.W#         shortcut = x#         x = self.norm1(x)#         x = x.view(B, H, W, C)#         pad_l = pad_t = 0#         pad_r = (self.window_size - W % self.window_size) % self.window_size#         pad_b = (self.window_size - H % self.window_size) % self.window_size#         x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b))#         _, Hp, Wp, _ = x.shape#         if self.shift_size > 0:#             shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))#             attn_mask = mask_matrix#         else:#             shifted_x = x#             attn_mask = None#         x_windows = window_partition(shifted_x, self.window_size)#         x_windows = x_windows.view(-1, self.window_size * self.window_size, C)#         attn_windows = self.attn(x_windows, mask=attn_mask)#         attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)#         shifted_x = window_reverse(attn_windows, self.window_size, Hp, Wp)#         if self.shift_size > 0:#             x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))#         else:#             x = shifted_x#         if pad_r > 0 or pad_b > 0:#             x = x[:, :H, :W, :].contiguous()#         x = x.view(B, H * W, C)#         x = shortcut + self.drop_path(x)#         x = x + self.drop_path(self.mlp(self.norm2(x)))#         return x# class PatchMerging(nn.Module):#     """Downsample between stages: 2x spatial reduction, 2x channel increase."""#     def __init__(self, dim, norm_layer=nn.LayerNorm):#         super().__init__()#         self.dim = dim#         self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)#         self.norm = norm_layer(4 * dim)#     def forward(self, x, H, W):#         B, L, C = x.shape#         x = x.view(B, H, W, C)#         pad_input = (H % 2 == 1) or (W % 2 == 1)#         if pad_input:#             x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))#         x0 = x[:, 0::2, 0::2, :]#         x1 = x[:, 1::2, 0::2, :]#         x2 = x[:, 0::2, 1::2, :]#         x3 = x[:, 1::2, 1::2, :]#         x = torch.cat([x0, x1, x2, x3], -1)#         x = x.view(B, -1, 4 * C)#         x = self.norm(x)#         x = self.reduction(x)#         return x# class PatchEmbed(nn.Module):#     """Image to patch embedding."""#     def __init__(self, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):#         super().__init__()#         patch_size = to_2tuple(patch_size)#         self.patch_size = patch_size#         self.in_chans = in_chans#         self.embed_dim = embed_dim#         self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)#         self.norm = norm_layer(embed_dim) if norm_layer else None#     def forward(self, x):#         _, _, H, W = x.size()#         if W % self.patch_size[1] != 0:#             x = F.pad(x, (0, self.patch_size[1] - W % self.patch_size[1]))#         if H % self.patch_size[0] != 0:#             x = F.pad(x, (0, 0, 0, self.patch_size[0] - H % self.patch_size[0]))#         x = self.proj(x)#         if self.norm is not None:#             Wh, Ww = x.size(2), x.size(3)#             x = x.flatten(2).transpose(1, 2)#             x = self.norm(x)#             x = x.transpose(1, 2).view(-1, self.embed_dim, Wh, Ww)#         return x# print('Swin building blocks defined.')print('Swin blocks: COMMENTED OUT (using ConvNeXtV2 backbone)')

In [ ]:
# ============================================================# 5b. PWAM — Pixel-Word Attention Module# ============================================================##  ╔══════════════════════════════════════════════════════════════╗#  ║  ★★★ GNN REPLACEMENT TARGET ★★★                            ║#  ║                                                            ║#  ║  This module performs cross-modal fusion between visual    ║#  ║  features (from Swin) and language features (from BERT).  ║#  ║                                                            ║#  ║  Current: Scaled dot-product attention (query=visual,     ║#  ║           key/value=language), then element-wise multiply. ║#  ║                                                            ║#  ║  Future: Replace with GNN variant from Section 10.        ║#  ║  The GNN should have the SAME interface:                  ║#  ║    Input:  x (B, H*W, dim), l (B, dim_l, N_l),           ║#  ║            l_mask (B, N_l, 1)                             ║#  ║    Output: fused (B, H*W, dim)                            ║#  ╚══════════════════════════════════════════════════════════════╝#class SpatialImageLanguageAttention(nn.Module):    """    Cross-modal attention: visual queries attend to language keys/values.    This is the CORE fusion mechanism inside PWAM.    Input shapes:        x:      (B, H*W, v_in_channels)   — visual features        l:      (B, l_in_channels, N_l)    — language features (channel-first!)        l_mask: (B, N_l, 1)                — language attention mask    Output: (B, H*W, value_channels)    """    def __init__(self, v_in_channels, l_in_channels, key_channels,                 value_channels, out_channels=None, num_heads=1):        super().__init__()        self.v_in_channels = v_in_channels        self.l_in_channels = l_in_channels        self.out_channels = out_channels or value_channels        self.key_channels = key_channels        self.value_channels = value_channels        self.num_heads = num_heads        # Keys: from language (B, l_in, N_l) -> (B, key_ch, N_l)        self.f_key = nn.Sequential(            nn.Conv1d(l_in_channels, key_channels, 1, 1),        )        # Queries: from visual (B, v_in, H*W) -> (B, key_ch, H*W)        self.f_query = nn.Sequential(            nn.Conv1d(v_in_channels, key_channels, 1, 1),            nn.InstanceNorm1d(key_channels),        )        # Values: from language        self.f_value = nn.Sequential(            nn.Conv1d(l_in_channels, value_channels, 1, 1),        )        # Output projection        self.W = nn.Sequential(            nn.Conv1d(value_channels, self.out_channels, 1, 1),            nn.InstanceNorm1d(self.out_channels),        )    def forward(self, x, l, l_mask):        # x: (B, H*W, v_in)  l: (B, l_in, N_l)  l_mask: (B, N_l, 1)        B, HW = x.size(0), x.size(1)        x_t = x.permute(0, 2, 1)              # (B, v_in, H*W)        l_mask_t = l_mask.permute(0, 2, 1)     # (B, 1, N_l)        query = self.f_query(x_t).permute(0, 2, 1)  # (B, H*W, key_ch)        key   = self.f_key(l) * l_mask_t             # (B, key_ch, N_l)        value = self.f_value(l) * l_mask_t           # (B, val_ch, N_l)        n_l = value.size(-1)        # Multi-head reshape        query = query.reshape(B, HW, self.num_heads,                              self.key_channels // self.num_heads).permute(0, 2, 1, 3)        key   = key.reshape(B, self.num_heads,                            self.key_channels // self.num_heads, n_l)        value = value.reshape(B, self.num_heads,                              self.value_channels // self.num_heads, n_l)        # Scaled dot-product attention        sim = torch.matmul(query, key) * (self.key_channels ** -0.5)        sim = sim + (1e4 * l_mask_t.unsqueeze(1) - 1e4)  # mask padding        sim = F.softmax(sim, dim=-1)                       # (B, heads, H*W, N_l)        out = torch.matmul(sim, value.permute(0, 1, 3, 2))  # (B, heads, H*W, val//heads)        out = out.permute(0, 2, 1, 3).contiguous().reshape(B, HW, self.value_channels)        out = self.W(out.permute(0, 2, 1)).permute(0, 2, 1)  # (B, H*W, out_ch)        return outclass PWAM(nn.Module):    """    Pixel-Word Attention Module — the cross-modal fusion module.    ★★★ TO REPLACE WITH GNN: ★★★    1. Keep the same __init__ signature (dim, v_in, l_in, key, value, heads, dropout)    2. Keep the same forward signature: (x, l, l_mask) -> (B, H*W, dim)    3. Replace self.image_lang_att with your GNN module    4. Optionally keep or modify vis_project and project_mm    """    def __init__(self, dim, v_in_channels, l_in_channels,                 key_channels, value_channels, num_heads=0, dropout=0.0):        super().__init__()        self.vis_project = nn.Sequential(            nn.Conv1d(dim, dim, 1, 1),            nn.GELU(),            nn.Dropout(dropout),        )        self.image_lang_att = SpatialImageLanguageAttention(            v_in_channels, l_in_channels, key_channels, value_channels,            out_channels=value_channels, num_heads=num_heads)        self.project_mm = nn.Sequential(            nn.Conv1d(value_channels, value_channels, 1, 1),            nn.GELU(),            nn.Dropout(dropout),        )    def forward(self, x, l, l_mask):        # x: (B, H*W, dim)  l: (B, 768, N_l)  l_mask: (B, N_l, 1)        vis  = self.vis_project(x.permute(0, 2, 1))  # (B, dim, H*W)        lang = self.image_lang_att(x, l, l_mask)      # (B, H*W, dim)        lang = lang.permute(0, 2, 1)                   # (B, dim, H*W)        mm = torch.mul(vis, lang)                       # element-wise fusion        mm = self.project_mm(mm)                        # (B, dim, H*W)        return mm.permute(0, 2, 1)                      # (B, H*W, dim)print('PWAM (Pixel-Word Attention Module) defined.')print('  -> This is the fusion module to replace with GNN.')

In [ ]:
# ============================================================# 5b-ii. GNN Fusion Modules (same interface as PWAM)# ============================================================# These are the EXACT same implementations from Section 13 of the# original notebook, now positioned before MMBasicLayer so they# can be used via build_fusion_module().class GCNFusion(nn.Module):    """GCN-based V-L fusion. Same interface as PWAM."""    def __init__(self, dim, v_in_channels, l_in_channels,                 key_channels, value_channels, num_heads=1, dropout=0.0):        super().__init__()        self.k = 8        self.lang_proj = nn.Sequential(            nn.Conv1d(l_in_channels, dim, 1), nn.GELU(), nn.Dropout(dropout))        self.gcn1 = nn.Linear(dim, dim)        self.gcn2 = nn.Linear(dim, dim)        self.norm1 = nn.LayerNorm(dim)        self.norm2 = nn.LayerNorm(dim)        self.cross_attn = nn.MultiheadAttention(dim, max(num_heads, 1), batch_first=True, dropout=dropout)        self.out_proj = nn.Sequential(            nn.Conv1d(dim, dim, 1), nn.GELU(), nn.Dropout(dropout))    def forward(self, x, l, l_mask):        B, N, C = x.shape        with torch.no_grad():            xn = F.normalize(x, dim=-1)            sim = torch.bmm(xn, xn.transpose(1,2))            topk_v, topk_i = sim.topk(self.k, dim=-1)            adj = torch.zeros_like(sim)            adj.scatter_(-1, topk_i, topk_v)            deg = adj.sum(-1, keepdim=True).clamp(min=1e-6)            adj = adj / deg        h = self.norm1(F.relu(self.gcn1(torch.bmm(adj, x))))        h = self.norm2(F.relu(self.gcn2(torch.bmm(adj, h))))        lang = self.lang_proj(l).permute(0, 2, 1)        kpm = (l_mask.squeeze(-1) == 0) if l_mask is not None else None        h, _ = self.cross_attn(h, lang, lang, key_padding_mask=kpm)        return self.out_proj(h.permute(0,2,1)).permute(0,2,1)class CrossModalGNN(nn.Module):    """Two edge types: V->V + V<->L. Same interface as PWAM."""    def __init__(self, dim, v_in_channels, l_in_channels,                 key_channels, value_channels, num_heads=1, dropout=0.0):        super().__init__()        heads = max(num_heads, 1)        self.rounds = 2        self.lang_proj = nn.Sequential(            nn.Conv1d(l_in_channels, dim, 1), nn.GELU(), nn.Dropout(dropout))        self.vv_attn = nn.ModuleList([            nn.MultiheadAttention(dim, heads, batch_first=True, dropout=dropout)            for _ in range(self.rounds)])        self.vv_norm = nn.ModuleList([nn.LayerNorm(dim) for _ in range(self.rounds)])        self.vl_attn = nn.ModuleList([            nn.MultiheadAttention(dim, heads, batch_first=True, dropout=dropout)            for _ in range(self.rounds)])        self.vl_norm = nn.ModuleList([nn.LayerNorm(dim) for _ in range(self.rounds)])        self.ffn = nn.ModuleList([            nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(),                          nn.Linear(dim*4, dim), nn.LayerNorm(dim))            for _ in range(self.rounds)])        self.out_proj = nn.Sequential(            nn.Conv1d(dim, dim, 1), nn.GELU(), nn.Dropout(dropout))    def forward(self, x, l, l_mask):        lang = self.lang_proj(l).permute(0, 2, 1)        kpm = (l_mask.squeeze(-1) == 0) if l_mask is not None else None        h = x        for i in range(self.rounds):            vv, _ = self.vv_attn[i](h, h, h)            h = self.vv_norm[i](h + vv)            vl, _ = self.vl_attn[i](h, lang, lang, key_padding_mask=kpm)            h = self.vl_norm[i](h + vl)            h = h + self.ffn[i](h)        return self.out_proj(h.permute(0,2,1)).permute(0,2,1)class GraphSAGEFusion(nn.Module):    """GraphSAGE-style aggregation + language gating. Same interface as PWAM."""    def __init__(self, dim, v_in_channels, l_in_channels,                 key_channels, value_channels, num_heads=1, dropout=0.0):        super().__init__()        self.k = 8        self.lang_proj = nn.Sequential(            nn.Conv1d(l_in_channels, dim, 1), nn.GELU(), nn.Dropout(dropout))        self.sage1_self  = nn.Linear(dim, dim)        self.sage1_neigh = nn.Linear(dim, dim)        self.sage2_self  = nn.Linear(dim, dim)        self.sage2_neigh = nn.Linear(dim, dim)        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)        self.gate = nn.Sequential(            nn.Linear(dim*2, dim), nn.ReLU(),            nn.Linear(dim, dim), nn.Sigmoid())        self.out_proj = nn.Sequential(            nn.Conv1d(dim, dim, 1), nn.GELU(), nn.Dropout(dropout))    def forward(self, x, l, l_mask):        B, N, C = x.shape        with torch.no_grad():            sim = torch.bmm(F.normalize(x, dim=-1),                            F.normalize(x, dim=-1).transpose(1,2))            _, idx = sim.topk(self.k+1, dim=-1)            idx = idx[:, :, 1:]        ix = idx.unsqueeze(-1).expand(-1,-1,-1,C)        def gather_mean(h):            nb = torch.gather(h.unsqueeze(2).expand(-1,-1,self.k,-1), 1, ix)            return nb.mean(dim=2)        h = self.n1(F.relu(self.sage1_self(x) + self.sage1_neigh(gather_mean(x))))        h = self.n2(F.relu(self.sage2_self(h) + self.sage2_neigh(gather_mean(h))))        lp = self.lang_proj(l).permute(0,2,1)        if l_mask is not None:            lpool = (lp * l_mask).sum(1) / l_mask.sum(1).clamp(min=1)        else:            lpool = lp.mean(1)        g = self.gate(torch.cat([h, lpool.unsqueeze(1).expand_as(h)], -1))        out = g * h        return self.out_proj(out.permute(0,2,1)).permute(0,2,1)# ── Factory function to select fusion module ──def build_fusion_module(fusion_type, dim, num_heads_fusion=1, fusion_drop=0.0):    """Build the appropriate fusion module based on config."""    if fusion_type == 'gcn':        return GCNFusion(dim, dim, 768, dim, dim, num_heads_fusion, fusion_drop)    elif fusion_type == 'cross_modal_gnn':        return CrossModalGNN(dim, dim, 768, dim, dim, num_heads_fusion, fusion_drop)    elif fusion_type == 'graphsage':        return GraphSAGEFusion(dim, dim, 768, dim, dim, num_heads_fusion, fusion_drop)    else:  # 'pwam' or default        return PWAM(dim, dim, 768, dim, dim, num_heads=num_heads_fusion, dropout=fusion_drop)print('GNN fusion modules defined: GCNFusion, CrossModalGNN, GraphSAGEFusion')print(f'build_fusion_module() ready — current selection: {cfg.fusion_type}')

In [ ]:
# ============================================================# 5b-iii. Quick test: verify all fusion modules match PWAM interface# ============================================================def test_fusion(cls, name):    B, N, C, Nl, Cl = 2, 225, 128, 20, 768    m = cls(dim=C, v_in_channels=C, l_in_channels=Cl,            key_channels=C, value_channels=C, num_heads=1)    x = torch.randn(B, N, C)    l = torch.randn(B, Cl, Nl)    lm = torch.ones(B, Nl, 1); lm[:, 15:, :] = 0    out = m(x, l, lm)    assert out.shape == (B, N, C), f'{name}: expected {(B,N,C)}, got {out.shape}'    print(f'  {name:20s} OK -> {out.shape}')print('Testing fusion modules...')test_fusion(PWAM, 'PWAM (baseline)')test_fusion(GCNFusion, 'GCNFusion')test_fusion(CrossModalGNN, 'CrossModalGNN')test_fusion(GraphSAGEFusion, 'GraphSAGEFusion')print('All modules pass!')

In [ ]:
# ============================================================# 5c. MMBasicLayer — Swin stage with fusion [COMMENTED OUT]# ============================================================# This is only needed for the Swin backbone path.# ConvNeXtV2 uses MMConvNeXtV2Stage instead.# # ============================================================# # 5c. MMBasicLayer — one Swin stage with PWAM fusion# # ============================================================# class MMBasicLayer(nn.Module):#     """One stage of the MultiModal Swin Transformer."""#     def __init__(self, dim, depth, num_heads, window_size=7,#                  mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0.,#                  attn_drop=0., drop_path=0., norm_layer=nn.LayerNorm,#                  downsample=None, use_checkpoint=False,#                  num_heads_fusion=1, fusion_drop=0.0):#         super().__init__()#         self.window_size = window_size#         self.shift_size = window_size // 2#         self.depth = depth#         self.use_checkpoint = use_checkpoint#         self.dim = dim#         self.blocks = nn.ModuleList([#             SwinTransformerBlock(#                 dim=dim, num_heads=num_heads, window_size=window_size,#                 shift_size=0 if (i % 2 == 0) else window_size // 2,#                 mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,#                 drop=drop, attn_drop=attn_drop,#                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,#                 norm_layer=norm_layer)#             for i in range(depth)#         ])#         self.fusion = PWAM(#             dim, dim, 768, dim, dim,#             num_heads=num_heads_fusion,#             dropout=fusion_drop#         )#         self.res_gate = nn.Sequential(#             nn.Linear(dim, dim, bias=False), nn.ReLU(),#             nn.Linear(dim, dim, bias=False), nn.Tanh(),#         )#         self.visual_residual = VariousReceptive(dim)#         self.visual_gate = nn.Sequential(#             nn.Linear(dim, dim, bias=False), nn.ReLU(),#             nn.Linear(dim, dim, bias=False), nn.Tanh(),#         )#         self.downsample = downsample(dim=dim, norm_layer=norm_layer) if downsample else None#         # Cached attention mask — computed once, reused every batch#         self._cached_mask_key = None#         self._cached_attn_mask = None#     def _get_attn_mask(self, H, W, device):#         key = (H, W, device)#         if self._cached_mask_key == key and self._cached_attn_mask is not None:#             return self._cached_attn_mask#         Hp = math.ceil(H / self.window_size) * self.window_size#         Wp = math.ceil(W / self.window_size) * self.window_size#         img_mask = torch.zeros((1, Hp, Wp, 1), device=device)#         h_slices = (slice(0, -self.window_size),#                     slice(-self.window_size, -self.shift_size),#                     slice(-self.shift_size, None))#         w_slices = (slice(0, -self.window_size),#                     slice(-self.window_size, -self.shift_size),#                     slice(-self.shift_size, None))#         cnt = 0#         for h in h_slices:#             for w in w_slices:#                 img_mask[:, h, w, :] = cnt#                 cnt += 1#         mask_windows = window_partition(img_mask, self.window_size)#         mask_windows = mask_windows.view(-1, self.window_size * self.window_size)#         attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)#         attn_mask = attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)#         self._cached_mask_key = key#         self._cached_attn_mask = attn_mask#         return attn_mask#     def forward(self, x, H, W, l, l_mask):#         attn_mask = self._get_attn_mask(H, W, x.device)#         for blk in self.blocks:#             blk.H, blk.W = H, W#             if self.use_checkpoint:#                 x = checkpoint_util.checkpoint(blk, x, attn_mask, use_reentrant=False)#             else:#                 x = blk(x, attn_mask)#         v_residual = self.visual_residual(x)#         x_residual = self.fusion(x, l, l_mask)#         x = x + (self.res_gate(x_residual) * x_residual) + \#                 (self.visual_gate(v_residual) * v_residual)#         if self.downsample is not None:#             x_down = self.downsample(x, H, W)#             Wh, Ww = (H + 1) // 2, (W + 1) // 2#             return x_residual, H, W, x_down, Wh, Ww#         else:#             return x_residual, H, W, x, H, W# print('MMBasicLayer defined (cached attention masks, gradient checkpointing).')print('MMBasicLayer: COMMENTED OUT (using ConvNeXtV2 backbone)')

In [ ]:
# ============================================================# 5d. MultiModalSwinTransformer [COMMENTED OUT]# ============================================================# # ============================================================# # 5d. MultiModalSwinTransformer — Full Backbone# # ============================================================# class MultiModalSwinTransformer(nn.Module):#     def __init__(self, pretrain_img_size=224, patch_size=4, in_chans=3,#                  embed_dim=96, depths=[2,2,6,2], num_heads=[3,6,12,24],#                  window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,#                  drop_rate=0., attn_drop_rate=0., drop_path_rate=0.2,#                  norm_layer=nn.LayerNorm, ape=False, patch_norm=True,#                  out_indices=(0,1,2,3), frozen_stages=-1, use_checkpoint=False,#                  num_heads_fusion=[1,1,1,1], fusion_drop=0.0):#         super().__init__()#         self.pretrain_img_size = pretrain_img_size#         self.num_layers = len(depths)#         self.embed_dim = embed_dim#         self.ape = ape#         self.patch_norm = patch_norm#         self.out_indices = out_indices#         self.frozen_stages = frozen_stages#         self.patch_embed = PatchEmbed(#             patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,#             norm_layer=norm_layer if patch_norm else None)#         if self.ape:#             pretrain_img_size = to_2tuple(pretrain_img_size)#             patch_size = to_2tuple(patch_size)#             patches_resolution = [#                 pretrain_img_size[0] // patch_size[0],#                 pretrain_img_size[1] // patch_size[1]#             ]#             self.absolute_pos_embed = nn.Parameter(#                 torch.zeros(1, embed_dim, patches_resolution[0], patches_resolution[1]))#             trunc_normal_(self.absolute_pos_embed, std=.02)#         self.pos_drop = nn.Dropout(p=drop_rate)#         dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]#         self.layers = nn.ModuleList()#         for i_layer in range(self.num_layers):#             layer = MMBasicLayer(#                 dim=int(embed_dim * 2 ** i_layer),#                 depth=depths[i_layer],#                 num_heads=num_heads[i_layer],#                 window_size=window_size,#                 mlp_ratio=mlp_ratio,#                 qkv_bias=qkv_bias, qk_scale=qk_scale,#                 drop=drop_rate, attn_drop=attn_drop_rate,#                 drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer+1])],#                 norm_layer=norm_layer,#                 downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,#                 use_checkpoint=use_checkpoint,#                 num_heads_fusion=num_heads_fusion[i_layer],#                 fusion_drop=fusion_drop,#             )#             self.layers.append(layer)#         num_features = [int(embed_dim * 2 ** i) for i in range(self.num_layers)]#         self.num_features = num_features#         for i_layer in out_indices:#             layer = norm_layer(num_features[i_layer])#             self.add_module(f'norm{i_layer}', layer)#         # Cross-scale Interaction Module (imported from repo)#         self.cim = CIM(dim=sum(num_features))#         self._freeze_stages()#     def _freeze_stages(self):#         if self.frozen_stages >= 0:#             self.patch_embed.eval()#             for param in self.patch_embed.parameters():#                 param.requires_grad = False#         if self.frozen_stages >= 1 and self.ape:#             self.absolute_pos_embed.requires_grad = False#         if self.frozen_stages >= 2:#             self.pos_drop.eval()#             for i in range(0, self.frozen_stages - 1):#                 m = self.layers[i]#                 m.eval()#                 for param in m.parameters():#                     param.requires_grad = False#     def init_weights(self):#         """Initialize weights (called before loading pretrained)."""#         def _init(m):#             if isinstance(m, nn.Linear):#                 trunc_normal_(m.weight, std=.02)#                 if m.bias is not None:#                     nn.init.constant_(m.bias, 0)#             elif isinstance(m, nn.LayerNorm):#                 nn.init.constant_(m.bias, 0)#                 nn.init.constant_(m.weight, 1.0)#         self.apply(_init)#     def forward(self, x, l, l_mask):#         """Forward pass.#         Args:#             x:      (B, 3, H, W) image#             l:      (B, 768, N_l) language features (channel-first)#             l_mask: (B, N_l, 1) language mask#         Returns:#             tuple of 4 multi-scale feature maps after CIM#         """#         x = self.patch_embed(x)#         Wh, Ww = x.size(2), x.size(3)#         if self.ape:#             ape = F.interpolate(self.absolute_pos_embed, size=(Wh, Ww), mode='bicubic')#             x = (x + ape).flatten(2).transpose(1, 2)#         else:#             x = x.flatten(2).transpose(1, 2)#         x = self.pos_drop(x)#         outs = []#         for i in range(self.num_layers):#             x_out, H, W, x, Wh, Ww = self.layers[i](x, Wh, Ww, l, l_mask)#             if i in self.out_indices:#                 norm_layer = getattr(self, f'norm{i}')#                 x_out = norm_layer(x_out)#                 out = x_out.view(-1, H, W, self.num_features[i]).permute(0, 3, 1, 2).contiguous()#                 outs.append(out)#         outs = self.cim(outs)#         return tuple(outs)#     def train(self, mode=True):#         super().train(mode)#         self._freeze_stages()# print('MultiModalSwinTransformer defined.')print('MultiModalSwinTransformer: COMMENTED OUT (using ConvNeXtV2)')

In [ ]:
# ============================================================# 5e. MultiModalConvNeXtV2 — Drop-in Backbone for Ablation# ============================================================##  Drop-in replacement for MultiModalSwinTransformer.#  Same forward(x, l, l_mask) → tuple(4 feature maps after CIM)#  Same channel dims [128, 256, 512, 1024] for Base variant.##  Architecture per stage (mirrors MMBasicLayer exactly):#    [Downsample] → ConvNeXtV2 blocks → PWAM → VariousReceptive#    → Gated residual → [next stage]##  Pretrained: facebook/convnextv2-base-22k-224##  Regularization:#    - Stochastic depth via cfg.drop_path (applied to ConvNeXtV2 blocks)#    - Stage freezing via cfg.frozen_stages (blocks only, fusion trains)#    - LLRD via cfg.layer_decay (handled in build_optimizer)# ============================================================class MMConvNeXtV2Stage(nn.Module):    """One stage of MultiModal ConvNeXt-V2.    Mirrors MMBasicLayer: same PWAM, VariousReceptive, and gating    logic, but with ConvNeXtV2 blocks instead of Swin blocks.    """    def __init__(self, convnext_blocks, dim,                 num_heads_fusion=1, fusion_drop=0.0,                 use_checkpoint=False, fusion_type='pwam'):        super().__init__()        self.blocks = convnext_blocks        self.dim = dim        self.use_checkpoint = use_checkpoint        # Cross-modal fusion (configurable via fusion_type)        self.fusion = build_fusion_module(            fusion_type, dim,            num_heads_fusion=num_heads_fusion,            fusion_drop=fusion_drop,        )        self.res_gate = nn.Sequential(            nn.Linear(dim, dim, bias=False), nn.ReLU(),            nn.Linear(dim, dim, bias=False), nn.Tanh(),        )        self.visual_residual = VariousReceptive(dim)        self.visual_gate = nn.Sequential(            nn.Linear(dim, dim, bias=False), nn.ReLU(),            nn.Linear(dim, dim, bias=False), nn.Tanh(),        )    def forward(self, x, l, l_mask):        """        Args:            x:      (B, C, H, W)  visual features (channels-first)            l:      (B, 768, N_l) language features            l_mask: (B, N_l, 1)   language mask        Returns:            x_residual: (B, H*W, C)  — PWAM output (for decoder)            H, W:       int          — spatial dims            x_out:      (B, C, H, W) — gated output (feeds next stage)        """        # ── ConvNeXtV2 blocks ──        for blk in self.blocks:            if self.use_checkpoint:                x = checkpoint_util.checkpoint(blk, x, use_reentrant=False)            else:                x = blk(x)        B, C, H, W = x.shape        x_flat = x.flatten(2).transpose(1, 2)           # (B, H*W, C)        # ── Cross-modal fusion (same gating as MMBasicLayer) ──        v_residual = self.visual_residual(x_flat)        x_residual = self.fusion(x_flat, l, l_mask)      # (B, H*W, C)        x_fused = x_flat + (self.res_gate(x_residual) * x_residual) + \                           (self.visual_gate(v_residual) * v_residual)        x_out = x_fused.transpose(1, 2).view(B, C, H, W)        return x_residual, H, W, x_outclass MultiModalConvNeXtV2(nn.Module):    """Full multi-modal ConvNeXt-V2 backbone.    Architecture (mirrors MultiModalSwinTransformer):        Image → Stem(4x↓) → [Stage0 → Stage1 → Stage2 → Stage3] → CIM        Each stage: [Downsample] → blocks → PWAM fusion → gating    Channel dims for Base: [128, 256, 512, 1024] — identical to Swin-Base.    """    def __init__(self, pretrained_model_name='facebook/convnextv2-base-22k-224',                 out_indices=(0, 1, 2, 3),                 use_checkpoint=False,                 num_heads_fusion=[1, 1, 1, 1],                 fusion_drop=0.0,                 drop_path=0.0,                 frozen_stages=-1,                 fusion_type='pwam'):        super().__init__()        self.out_indices = out_indices        self.frozen_stages = frozen_stages        # ── Load pretrained ConvNeXt-V2 with drop_path override ──        print(f'Loading ConvNeXt-V2 from: {pretrained_model_name}')        hf_config = ConvNextV2Config.from_pretrained(pretrained_model_name)        hf_config.drop_path_rate = drop_path        convnext = ConvNextV2Model.from_pretrained(            pretrained_model_name, config=hf_config)        self.hidden_sizes  = hf_config.hidden_sizes       # [128, 256, 512, 1024]        self.num_features  = self.hidden_sizes             # compatibility alias        # Stem (patch embedding: Conv2d k=4 s=4 + LayerNorm)        self.stem = convnext.embeddings        # Decompose each HF stage into downsampling + blocks.        self.downsample_layers = nn.ModuleList()        self.mm_stages = nn.ModuleList()        for i, hf_stage in enumerate(convnext.encoder.stages):            # Downsampling (Identity for stage 0, LN+Conv2d↓2 for 1-3)            if len(hf_stage.downsampling_layer) > 0:                self.downsample_layers.append(                    nn.Sequential(*list(hf_stage.downsampling_layer)))            else:                self.downsample_layers.append(nn.Identity())            # Blocks + fusion            self.mm_stages.append(MMConvNeXtV2Stage(                convnext_blocks=hf_stage.layers,                dim=self.hidden_sizes[i],                num_heads_fusion=num_heads_fusion[i],                fusion_drop=fusion_drop,                use_checkpoint=use_checkpoint and (i == 2),                fusion_type=fusion_type,            ))        # Output norms        for i in out_indices:            self.add_module(f'norm{i}', nn.LayerNorm(self.hidden_sizes[i]))        # Cross-scale Interaction Module        self.cim = CIM(dim=sum(self.hidden_sizes))        del convnext        # ── Freeze early stages ──        self._freeze_stages()        # ── Report ──        frozen_params = sum(p.numel() for p in self.parameters() if not p.requires_grad)        total_params  = sum(p.numel() for p in self.parameters())        print(f'  Channels: {self.hidden_sizes} | Depths: {list(hf_config.depths)}')        print(f'  drop_path: {drop_path} | frozen_stages: {frozen_stages}')        print(f'  Frozen: {frozen_params:,} / {total_params:,} backbone params')    def _freeze_stages(self):        """Freeze stem and early ConvNeXt blocks (NOT fusion modules).        frozen_stages=-1: nothing frozen        frozen_stages=0:  stem frozen        frozen_stages=1:  stem + stages 0-1 blocks + downsample layers frozen        frozen_stages=2:  stem + stages 0-2 blocks + downsample layers frozen        PWAM, VariousReceptive, gates, norms, and CIM always train.        """        if self.frozen_stages >= 0:            # Freeze stem            self.stem.eval()            for p in self.stem.parameters():                p.requires_grad = False        # Freeze ConvNeXt blocks + downsample in early stages        for i in range(min(self.frozen_stages + 1, len(self.mm_stages))):            # Downsample layer            self.downsample_layers[i].eval()            for p in self.downsample_layers[i].parameters():                p.requires_grad = False            # ConvNeXt blocks only — fusion modules stay trainable            for p in self.mm_stages[i].blocks.parameters():                p.requires_grad = False    def forward(self, x, l, l_mask):        """Same signature as MultiModalSwinTransformer.forward()."""        x = self.stem(x)                                  # (B, 128, H/4, W/4)        outs = []        for i in range(len(self.mm_stages)):            x = self.downsample_layers[i](x)            x_residual, H, W, x = self.mm_stages[i](x, l, l_mask)            if i in self.out_indices:                norm_layer = getattr(self, f'norm{i}')                x_out = norm_layer(x_residual)                out = x_out.view(-1, H, W, self.hidden_sizes[i]) \                           .permute(0, 3, 1, 2).contiguous()                outs.append(out)        outs = self.cim(outs)        return tuple(outs)    def train(self, mode=True):        """Override to keep frozen stages in eval mode."""        super().train(mode)        self._freeze_stages()print('MultiModalConvNeXtV2 defined.')print('  -> Same forward(x, l, l_mask) as MultiModalSwinTransformer')print('  -> Channels [128, 256, 512, 1024] — decoder/CIM/PWAM unchanged')

---## 6. Decoder — SimpleDecoding

In [ ]:
# ============================================================# 6. SimpleDecoding with optional Adaptive Rotated Conv# ============================================================def make_arc_or_conv(in_ch, out_ch, ks=3, pad=1, bias=False):    """Use ARC if available, else standard Conv2d."""    try:        rf = RountingFunction(in_channels=in_ch, kernel_number=1)        return AdaptiveRotatedConv2d(            in_channels=in_ch, out_channels=out_ch,            kernel_size=ks, padding=pad,            rounting_func=rf, bias=bias, kernel_number=1)    except Exception:        return nn.Conv2d(in_ch, out_ch, ks, padding=pad, bias=bias)class SimpleDecoding(nn.Module):    """FPN-style top-down decoder with ARC."""    def __init__(self, c4_dims, factor=2):        super().__init__()        h = c4_dims // factor        c3 = c4_dims // (factor**1)        c2 = c4_dims // (factor**2)        c1 = c4_dims // (factor**3)        # Stage 4+3        self.conv1_4 = nn.Conv2d(c4_dims + c3, h, 3, padding=1, bias=False)        self.conv2_4 = make_arc_or_conv(h, h)        self.bn1_4, self.bn2_4 = nn.BatchNorm2d(h), nn.BatchNorm2d(h)        self.relu1_4, self.relu2_4 = nn.ReLU(), nn.ReLU()        # Stage +2        self.conv1_3 = nn.Conv2d(h + c2, h, 3, padding=1, bias=False)        self.conv2_3 = make_arc_or_conv(h, h)        self.bn1_3, self.bn2_3 = nn.BatchNorm2d(h), nn.BatchNorm2d(h)        self.relu1_3, self.relu2_3 = nn.ReLU(), nn.ReLU()        # Stage +1        self.conv1_2 = nn.Conv2d(h + c1, h, 3, padding=1, bias=False)        self.conv2_2 = make_arc_or_conv(h, h)        self.bn1_2, self.bn2_2 = nn.BatchNorm2d(h), nn.BatchNorm2d(h)        self.relu1_2, self.relu2_2 = nn.ReLU(), nn.ReLU()        # Head        self.head = nn.Conv2d(h, 2, 1)    def forward(self, x_c4, x_c3, x_c2, x_c1):        if x_c4.size(-2) < x_c3.size(-2) or x_c4.size(-1) < x_c3.size(-1):            x_c4 = F.interpolate(x_c4, scale_factor=2, mode='bilinear', align_corners=True)        x = self.relu1_4(self.bn1_4(self.conv1_4(torch.cat([x_c4, x_c3], 1))))        x = self.relu2_4(self.bn2_4(self.conv2_4(x)))        if x.size(-2) < x_c2.size(-2) or x.size(-1) < x_c2.size(-1):            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)        x = self.relu1_3(self.bn1_3(self.conv1_3(torch.cat([x, x_c2], 1))))        x = self.relu2_3(self.bn2_3(self.conv2_3(x)))        if x.size(-2) < x_c1.size(-2) or x.size(-1) < x_c1.size(-1):            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)        x = self.relu1_2(self.bn1_2(self.conv1_2(torch.cat([x, x_c1], 1))))        x = self.relu2_2(self.bn2_2(self.conv2_2(x)))        return self.head(x)print('SimpleDecoding defined.')

---## 7. Model Assembly

In [ ]:
# ============================================================# 7a. Open-Set Network Components (used when cfg.USE_OPEN_SET=True)# ============================================================class TextVisualAlignmentGate(nn.Module):    """Learns alignment score between text and visual features.    Low alignment → likely out-of-distribution (mismatched text)."""    def __init__(self, vis_dim=1024, text_dim=768, hidden=256):        super().__init__()        self.vis_proj = nn.Linear(vis_dim, hidden)        self.txt_proj = nn.Linear(text_dim, hidden)        self.gate = nn.Sequential(            nn.Linear(hidden * 2, hidden), nn.ReLU(),            nn.Linear(hidden, 1), nn.Sigmoid())    def forward(self, vis_feat, text_feat):        # vis_feat: (B, vis_dim)  text_feat: (B, text_dim)        v = self.vis_proj(vis_feat)        t = self.txt_proj(text_feat)        return self.gate(torch.cat([v, t], dim=-1))  # (B, 1)class EnergyBasedOOD(nn.Module):    """Energy-based out-of-distribution detector."""    def __init__(self, temperature=1.0):        super().__init__()        self.T = temperature    def energy_score(self, logits):        # logits: (B, 2, H, W)        return -self.T * torch.logsumexp(logits / self.T, dim=1)  # (B, H, W)    def forward(self, logits):        energy = self.energy_score(logits)        return energy.mean(dim=(1, 2))  # per-sample energy (B,)print('Open-set components defined (TextVisualAlignmentGate, EnergyBasedOOD).')

In [ ]:
# ============================================================# 7b. RMSINModel — with optional Open-Set head# ============================================================class RMSINModel(nn.Module):    def __init__(self, backbone, classifier, cfg):        super().__init__()        self.backbone = backbone        self.classifier = classifier        self.text_encoder = BertModel.from_pretrained(cfg.bert_type)        self.text_encoder.pooler = None        # Open-set components (always created, only used when enabled)        vis_dim = backbone.hidden_sizes[-1] if hasattr(backbone, 'hidden_sizes') else 1024        self.alignment_gate = TextVisualAlignmentGate(vis_dim=vis_dim)        self.ood_detector = EnergyBasedOOD(temperature=1.0)    def forward(self, image, sentences, attentions, return_alignment=False):        # Text encoding        bert_out = self.text_encoder(input_ids=sentences, attention_mask=attentions)        l_feats = bert_out.last_hidden_state.permute(0, 2, 1)  # (B, 768, N_l)        l_mask  = attentions.unsqueeze(-1).float()              # (B, N_l, 1)        # Vision backbone with cross-modal fusion        features = self.backbone(image, l_feats, l_mask)        # Decode        x_c1, x_c2, x_c3, x_c4 = features        logits = self.classifier(x_c4, x_c3, x_c2, x_c1)        if return_alignment:            # Global average pool of deepest features for alignment            vis_global = F.adaptive_avg_pool2d(x_c4, 1).flatten(1)  # (B, C)            txt_global = bert_out.last_hidden_state[:, 0, :]         # (B, 768) [CLS]            alignment = self.alignment_gate(vis_global, txt_global)  # (B, 1)            energy = self.ood_detector(logits)                        # (B,)            return logits, alignment, energy        return logitsprint('RMSINModel defined (with optional open-set outputs).')

In [ ]:
# ============================================================# 7c. Build model (ConvNeXtV2 + configurable fusion)# ============================================================def build_model(cfg):    backbone_type = getattr(cfg, 'backbone', 'convnextv2')    if backbone_type == 'convnextv2':        convnext_name = getattr(cfg, 'convnextv2_model',                                'facebook/convnextv2-base-22k-224')        backbone = MultiModalConvNeXtV2(            pretrained_model_name=convnext_name,            out_indices=(0, 1, 2, 3),            use_checkpoint=True,            num_heads_fusion=cfg.mha_list,            fusion_drop=cfg.fusion_drop,            drop_path=cfg.drop_path,            frozen_stages=cfg.frozen_stages,            fusion_type=cfg.fusion_type,           # ← configurable        )        decoder_dim = backbone.hidden_sizes[-1]    # 1024    else:        raise ValueError(            f'backbone={backbone_type} not supported. '            f'This notebook uses ConvNeXtV2 only. '            f'Swin code is commented out.')    classifier = SimpleDecoding(decoder_dim)    model      = RMSINModel(backbone, classifier, cfg)    # Freeze BERT layers 10 & 11 + embeddings    for i in [10, 11]:        for p in model.text_encoder.encoder.layer[i].parameters():            p.requires_grad = False    for p in model.text_encoder.embeddings.parameters():        p.requires_grad = False    total     = sum(p.numel() for p in model.parameters())    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)    frozen    = total - trainable    print(f'Backbone: {backbone_type} | Fusion: {cfg.fusion_type}')    print(f'Total: {total:,} | Trainable: {trainable:,} | Frozen: {frozen:,}')    return modelmodel = build_model(cfg).to(DEVICE)print(f'Model on: {next(model.parameters()).device}')

In [ ]:
# ============================================================# 7c. Model Architecture Printer# ============================================================def print_model_architecture(model):    # ── Top-level module breakdown ───────────────────────────    print("\n" + "="*70)    print("  LEVEL 1 — Top-Level Module Breakdown")    print("="*70)    rows = []    for name, module in model.named_children():        n_params    = sum(p.numel() for p in module.parameters())        n_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)        rows.append([name, module.__class__.__name__,                     f"{n_params:,}", f"{n_trainable:,}",                     f"{n_params * 4 / 1024**2:.2f} MB"])    print(tabulate(rows,                   headers=["Module Name", "Class", "Total Params",                             "Trainable Params", "Approx Size (fp32)"],                   tablefmt="rounded_outline"))    # ── Grand total ──────────────────────────────────────────    total     = sum(p.numel() for p in model.parameters())    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)    frozen    = total - trainable    size_fp32 = total * 4 / 1024**2    size_fp16 = total * 2 / 1024**2    print("\n" + "="*70)    print("  GRAND TOTAL")    print("="*70)    print(tabulate([        ["Total parameters",     f"{total:,}"],        ["Trainable parameters", f"{trainable:,}"],        ["Frozen parameters",    f"{frozen:,}"],        ["Model size (fp32)",    f"{size_fp32:.2f} MB"],        ["Model size (fp16)",    f"{size_fp16:.2f} MB"],        ["Est. training VRAM*",  f"~{size_fp32 * 4:.0f} MB  (weights + grads + Adam)"],    ], tablefmt="rounded_outline"))    print("  * activations not included — depends on batch size & resolution\n")print_model_architecture(model)

---## 8. Loss Function

In [ ]:
# ============================================================# 8. Loss Function — Simple CE + Dice (baseline)# ============================================================# NOTE: Custom loss and imbalance strategies are tested in# the separate ablation notebook. This pipeline uses a clean# CE + Dice baseline.class DiceLoss(nn.Module):    def __init__(self, smooth=1e-6):        super().__init__()        self.smooth = smooth    def forward(self, pred, target):        C = pred.shape[1]        pred_soft = F.softmax(pred, dim=1)        target_oh = torch.stack([target == c for c in range(C)], dim=1).float()        inter = (pred_soft * target_oh).sum(dim=(2, 3))        union = (pred_soft + target_oh).sum(dim=(2, 3))        return (1 - (2 * inter + self.smooth) / (union + self.smooth)).mean()class CEDiceLoss(nn.Module):    def __init__(self, dice_weight=0.1, ce_weight=None):        super().__init__()        self.dice_weight = dice_weight        if ce_weight is not None:            self.register_buffer('ce_w', torch.FloatTensor(ce_weight))        else:            self.ce_w = None        self.dice = DiceLoss()    def forward(self, pred, target):        if pred.shape[-2:] != target.shape[-2:]:            pred = F.interpolate(pred, size=target.shape[-2:],                                 mode='bilinear', align_corners=False)        target_long = target.squeeze(1).long()        ce = F.cross_entropy(pred, target_long, weight=self.ce_w)        dice = self.dice(pred, target_long)        return (1 - self.dice_weight) * ce + self.dice_weight * dice# Knowledge Distillation lossclass DistillationLoss(nn.Module):    """KL-divergence between student and teacher logits."""    def __init__(self, temperature=4.0, alpha=0.5):        super().__init__()        self.T = temperature        self.alpha = alpha    def forward(self, student_logits, teacher_logits, target, task_criterion):        # Task loss        task_loss = task_criterion(student_logits, target)        # Distillation loss        s_log = F.log_softmax(student_logits / self.T, dim=1)        t_soft = F.softmax(teacher_logits / self.T, dim=1)        kd_loss = F.kl_div(s_log, t_soft, reduction='batchmean') * (self.T ** 2)        return (1 - self.alpha) * task_loss + self.alpha * kd_losscriterion = CEDiceLoss(dice_weight=0.1, ce_weight=[0.9, 1.1]).to(DEVICE)distill_criterion = DistillationLoss(    temperature=cfg.kd_temperature, alpha=cfg.kd_alpha).to(DEVICE) if cfg.USE_KD else Noneprint(f'Loss: CE(w=[0.9,1.1]) + 0.1*Dice')if cfg.USE_KD:    print(f'KD Loss: T={cfg.kd_temperature}, alpha={cfg.kd_alpha}')

---## 9. Evaluation Metrics

In [ ]:
# ============================================================# 9a. IoU metric# ============================================================def compute_iou(pred, target):    # pred: [B, 2, H, W] logits — upsample first, then argmax    if pred.shape[-2:] != target.shape[-2:]:        pred = F.interpolate(pred, size=target.shape[-2:],                             mode='bilinear', align_corners=False)    pred_mask  = pred.argmax(dim=1)                    # [B, H, W]    target_bin = target.squeeze(1).long()              # [B, H, W]    I = (pred_mask * target_bin).sum().item()    U = (pred_mask + target_bin).clamp(max=1).sum().item()    if U == 0:        return 1.0, 0, 0                               # empty mask edge case    return I / U, I, U

In [ ]:
# ============================================================# 9b. Segmentation Metrics  (oIoU, mIoU, Pr@K)# ============================================================class SegmentationMetrics:    """    Accumulates per-batch predictions and computes:      - oIoU  : overall IoU (global intersection / global union)      - mIoU  : mean of per-image IoU scores      - Pr@K  : fraction of images with per-image IoU >= K  (K in {0.5, 0.7, 0.9})    All inputs expected as binary tensors (0/1) on any device.    """    THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]    def __init__(self):        self.reset()    def reset(self):        self._total_inter = 0.0        self._total_union = 0.0        self._per_image_ious = []    def update(self, pred_mask: torch.Tensor, gt_mask: torch.Tensor):        """        pred_mask : (B, 1, H, W) or (B, H, W) — binary, float or bool        gt_mask   : same shape        """        pred = (pred_mask > 0.5).float().view(pred_mask.shape[0], -1)        gt   = (gt_mask   > 0.5).float().view(gt_mask.shape[0],   -1)        inter = (pred * gt).sum(dim=1)           # (B,)        union = (pred + gt).clamp(max=1).sum(dim=1)  # (B,)        # oIoU accumulation        self._total_inter += inter.sum().item()        self._total_union += union.sum().item()        # per-image IoU (avoid div-by-zero for empty gt)        per_iou = torch.where(union > 0,                              inter / union,                              torch.ones_like(inter))  # empty mask → IoU=1        self._per_image_ious.extend(per_iou.cpu().tolist())    def compute(self) -> dict:        oiou = self._total_inter / (self._total_union + 1e-8)        ious  = np.array(self._per_image_ious)        miou  = float(ious.mean())        results = {"oIoU": oiou, "mIoU": miou}        for k in self.THRESHOLDS:            results[f"Pr@{k}"] = float((ious >= k).mean())        return results

---## 10. Tracking & Logging

In [ ]:
# ============================================================# 10a. Time Trackers# ============================================================class EpochTimer:    """Tracks per-epoch training and validation wall-clock time."""    def __init__(self):        self._start = None        self.records = []          # list of dicts    def start(self):        self._start = time.perf_counter()    def stop(self, label="epoch"):        elapsed = time.perf_counter() - self._start        self.records.append({"label": label, "seconds": elapsed})        return elapsed    def summary(self):        if not self.records:            return        print("\n── Epoch Timing Summary ──────────────────────────────")        rows = []        for r in self.records:            rows.append([r["label"],                         f"{r['seconds']:.1f}s",                         str(datetime.timedelta(seconds=int(r['seconds'])))])        total = sum(r["seconds"] for r in self.records)        print(tabulate(rows, headers=["Phase", "Seconds", "H:M:S"], tablefmt="simple"))        print(f"  Total elapsed : {datetime.timedelta(seconds=int(total))}\n")class InferenceTimer:    """    Measures per-sample inference latency with GPU warm-up and CUDA sync.    Usage:        timer = InferenceTimer(model, device)        stats = timer.measure(val_loader, n_samples=100)        timer.report(stats)    """    def __init__(self, model, device):        self.model  = model        self.device = device    @torch.no_grad()    def measure(self, loader, n_samples=100):        self.model.eval()        latencies = []        # GPU warm-up — runs 5 batches without recording        for i, (img, msk, ids, attn) in enumerate(loader):            if i >= 5:                break            img, ids, attn = img.to(self.device), ids.to(self.device), attn.to(self.device)            _ = self.model(img, ids, attn)        if self.device.type == "cuda":            torch.cuda.synchronize()        # Timed loop        for i, (img, msk, ids, attn) in enumerate(loader):            if i >= n_samples:                break            img  = img.to(self.device)            ids  = ids.to(self.device)            attn = attn.to(self.device)            if self.device.type == "cuda":                torch.cuda.synchronize()            t0 = time.perf_counter()            _ = self.model(img, ids, attn)            if self.device.type == "cuda":                torch.cuda.synchronize()            latencies.append((time.perf_counter() - t0) * 1000)  # ms        return {            "n_samples"  : len(latencies),            "mean_ms"    : np.mean(latencies),            "std_ms"     : np.std(latencies),            "min_ms"     : np.min(latencies),            "max_ms"     : np.max(latencies),            "median_ms"  : np.median(latencies),            "fps"        : 1000.0 / np.mean(latencies),        }    def report(self, stats):        print("\n── Inference Latency (batch_size=1) ──────────────────")        rows = [            ["Samples measured",  stats["n_samples"]],            ["Mean latency",      f"{stats['mean_ms']:.2f} ms"],            ["Std deviation",     f"{stats['std_ms']:.2f} ms"],            ["Median latency",    f"{stats['median_ms']:.2f} ms"],            ["Min / Max",         f"{stats['min_ms']:.2f} ms / {stats['max_ms']:.2f} ms"],            ["Throughput",        f"{stats['fps']:.1f} FPS"],        ]        print(tabulate(rows, tablefmt="rounded_outline"))        print()

In [ ]:
# ============================================================# 10b. Epoch Logger# ============================================================class EpochLogger:    """    Stores per-epoch train/val loss and metrics, supports:      - dict-style access      - JSON export for reporting      - plotting loss + oIoU/mIoU curves in one call    """    def __init__(self, save_dir: str = "."):        self.save_dir = Path(save_dir)        self.save_dir.mkdir(parents=True, exist_ok=True)        self.history = defaultdict(list)   # key → list of values per epoch    def log(self, epoch: int, **kwargs):        """Call once per epoch with any key=value pairs."""        self.history["epoch"].append(epoch)        for k, v in kwargs.items():            self.history[k].append(float(v))        # Print a clean one-liner        parts = [f"Epoch {epoch:>3}"]        for k, v in kwargs.items():            parts.append(f"{k}: {v:.4f}")        print("  │  ".join(parts))    def save_json(self, filename="training_log.json"):        path = self.save_dir / filename        with open(path, "w") as f:            json.dump(dict(self.history), f, indent=2)        print(f"[Logger] History saved → {path}")    def plot_curves(self, filename="training_curves.png"):        epochs = self.history["epoch"]        fig, axes = plt.subplots(1, 3, figsize=(18, 5))        fig.suptitle("Training Curves", fontsize=14, fontweight="bold")        # ── Plot 1: Loss ──────────────────────────────────────        ax = axes[0]        if "train_loss" in self.history:            ax.plot(epochs, self.history["train_loss"], label="Train Loss",                    color="#2196F3", linewidth=2, marker="o", markersize=3)        if "val_loss" in self.history:            ax.plot(epochs, self.history["val_loss"],   label="Val Loss",                    color="#F44336", linewidth=2, marker="o", markersize=3)        ax.set_title("Loss (CE + Dice)")        ax.set_xlabel("Epoch")        ax.set_ylabel("Loss")        ax.legend()        ax.grid(True, alpha=0.3)        # ── Plot 2: IoU curves (equiv. of accuracy) ───────────        ax = axes[1]        for key, color, label in [            ("train_oIoU", "#2196F3", "Train oIoU"),            ("val_oIoU",   "#F44336", "Val oIoU"),            ("train_mIoU", "#4CAF50", "Train mIoU"),            ("val_mIoU",   "#FF9800", "Val mIoU"),        ]:            if key in self.history:                ax.plot(epochs, self.history[key], label=label,                        linewidth=2, marker="o", markersize=3)        ax.set_title("IoU Metrics (segmentation accuracy)")        ax.set_xlabel("Epoch")        ax.set_ylabel("IoU")        ax.set_ylim(0, 1)        ax.legend()        ax.grid(True, alpha=0.3)        # ── Plot 3: Precision @ thresholds ────────────────────        ax = axes[2]        threshold_styles = {            "val_Pr@0.5": ("#E91E63", "Pr@0.5"),            "val_Pr@0.7": ("#9C27B0", "Pr@0.7"),            "val_Pr@0.9": ("#607D8B", "Pr@0.9"),        }        for key, (color, label) in threshold_styles.items():            if key in self.history:                ax.plot(epochs, self.history[key], label=label,                        color=color, linewidth=2, marker="o", markersize=3)        ax.set_title("Val Precision @ IoU Thresholds")        ax.set_xlabel("Epoch")        ax.set_ylabel("Fraction of images")        ax.set_ylim(0, 1)        ax.legend()        ax.grid(True, alpha=0.3)        plt.tight_layout()        path = self.save_dir / filename        plt.savefig(path, dpi=150, bbox_inches="tight")        plt.show()        print(f"[Logger] Curves saved → {path}")

In [ ]:
# ============================================================# 10c. Checkpointing — stage-aware, Google Drive# ============================================================def _stage_paths(cfg):    """Generate stage-aware checkpoint filenames."""    base = f'{cfg.model_id}_{cfg.stage_name}'    return {        'last': os.path.join(cfg.output_dir, f'{base}_last.pth'),        'best': os.path.join(cfg.output_dir, f'{base}_best.pth'),    }def save_checkpoint(model, optimizer, scheduler, epoch, best_oiou, cfg, is_best=False):    paths = _stage_paths(cfg)    ckpt = {        'model'    : model.state_dict(),        'optimizer': optimizer.state_dict(),        'scheduler': scheduler.state_dict(),        'epoch'    : epoch,        'best_oiou': best_oiou,        'stage'    : cfg.stage_name,    }    torch.save(ckpt, paths['last'])    if is_best:        torch.save(ckpt, paths['best'])        print(f'  >> Best model saved! oIoU={best_oiou:.2f}%')    print(f'  Checkpoint: {paths["last"]}')def load_checkpoint(model, optimizer, scheduler, cfg):    paths = _stage_paths(cfg)    path = cfg.resume or paths['last']    if os.path.exists(path):        print(f'Resuming from {path}')        ckpt = torch.load(path, map_location=DEVICE)        model.load_state_dict(ckpt['model'], strict=False)        optimizer.load_state_dict(ckpt['optimizer'])        scheduler.load_state_dict(ckpt['scheduler'])        return ckpt['epoch'] + 1, ckpt.get('best_oiou', -1)    print(f'No checkpoint found for {cfg.stage_name}. Starting fresh.')    return 0, -1.0print(f'Checkpointing ready. Stage: {cfg.stage_name}')

In [ ]:
# ============================================================# 10d. Build KD Teacher (only for Stage 3)# ============================================================teacher_model = Noneif cfg.USE_KD:    # Teacher = Stage 2 best checkpoint    teacher_ckpt = os.path.join(        cfg.output_dir,        f'{cfg.model_id}_stage2_aug_openset_best.pth'    )    if not os.path.exists(teacher_ckpt):        raise FileNotFoundError(            f'KD requires Stage 2 best checkpoint at:\n  {teacher_ckpt}\n'            f'Train Stage 2 first (USE_AUGMENTATION=True, USE_OPEN_SET=True, USE_KD=False).'        )    print(f'Loading KD teacher from: {teacher_ckpt}')    teacher_model = build_model(cfg).to(DEVICE)    ckpt = torch.load(teacher_ckpt, map_location=DEVICE)    teacher_model.load_state_dict(ckpt['model'], strict=False)    teacher_model.eval()    for p in teacher_model.parameters():        p.requires_grad = False    print(f'Teacher loaded (epoch {ckpt["epoch"]}, oIoU={ckpt.get("best_oiou", "?")}%)')else:    print('KD disabled — no teacher model loaded.')

---## 11. Training Infrastructure & Loop

In [ ]:
# ============================================================# 11a. Build Optimizer and Scheduler (backbone-agnostic)# ============================================================def build_optimizer(model, cfg, num_batches):    # ── Layer-wise LR decay for ConvNeXt-V2 ──    # stage 0 (earliest, most general) → lowest LR    # stage 3 (deepest, most task-specific) → highest LR    llrd = getattr(cfg, 'layer_decay', 0.7)   # add to Config    backbone = model.backbone    param_groups = []    # Stem — very low LR    stem_params = [p for p in backbone.stem.parameters() if p.requires_grad]    if stem_params:        param_groups.append({            'params': stem_params,            'lr': cfg.lr * (llrd ** 4),            'weight_decay': cfg.weight_decay,        })    # Per-stage groups (stage 0 → lowest, stage 3 → near full LR)    num_stages = len(backbone.mm_stages)    for i in range(num_stages):        stage_lr = cfg.lr * (llrd ** (num_stages - 1 - i))        # ConvNeXt blocks        for name, p in backbone.mm_stages[i].blocks.named_parameters():            if not p.requires_grad:                continue            wd = 0.0 if any(kw in name.lower() for kw in                            {'norm', 'layernorm', 'bias', 'grn'}) else cfg.weight_decay            param_groups.append({                'params': [p], 'lr': stage_lr, 'weight_decay': wd,            })        # Fusion modules (PWAM, gates, VariousReceptive) — full LR        fusion_params = []        for n, p in backbone.mm_stages[i].named_parameters():            if not p.requires_grad:                continue            if 'blocks' not in n:   # everything except the convnext blocks                fusion_params.append(p)        if fusion_params:            param_groups.append({                'params': fusion_params, 'lr': cfg.lr, 'weight_decay': 0.0,            })    # Downsample layers, norms, CIM    other_bk = []    for name, p in backbone.named_parameters():        if not p.requires_grad:            continue        if 'mm_stages' not in name and 'stem' not in name:            other_bk.append(p)    if other_bk:        param_groups.append({            'params': other_bk, 'lr': cfg.lr, 'weight_decay': 0.0,        })    # Decoder    param_groups.append({        'params': [p for p in model.classifier.parameters() if p.requires_grad],        'lr': cfg.lr, 'weight_decay': cfg.weight_decay,    })    # BERT (layers 0-9 at reduced LR)    bert_params = []    for i in range(min(10, len(model.text_encoder.encoder.layer))):        bert_params.extend(p for p in model.text_encoder.encoder.layer[i].parameters()                           if p.requires_grad)    param_groups.append({'params': bert_params, 'lr': cfg.lr})    optimizer = torch.optim.AdamW(param_groups, lr=cfg.lr,                                  weight_decay=cfg.weight_decay, amsgrad=cfg.amsgrad)    total_steps = num_batches * cfg.epochs    scheduler = torch.optim.lr_scheduler.LambdaLR(        optimizer, lambda step: (1 - step / total_steps) ** 0.9)    return optimizer, scheduleroptimizer, scheduler = build_optimizer(model, cfg, len(train_loader))print(f'Optimizer: AdamW | LR={cfg.lr} | BERT LR={cfg.lr} | WD={cfg.weight_decay}')

In [ ]:
# ============================================================# 11b. Train one epoch (with open-set mismatch + KD support)# ============================================================def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch, cfg,                    metrics: SegmentationMetrics, timer: EpochTimer,                    teacher=None, distill_criterion=None):    model.train()    metrics.reset()    total_loss = 0.0    N = len(loader)    timer.start()    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', unit='batch',                dynamic_ncols=True, leave=True,                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')    for i, (imgs, masks, ids, attn) in enumerate(pbar):        imgs  = imgs.to(DEVICE, non_blocking=True)        masks = masks.to(DEVICE, non_blocking=True)        ids   = ids.to(DEVICE,  non_blocking=True)        attn  = attn.to(DEVICE, non_blocking=True)        # Open-set: randomly mismatch text for some samples        if cfg.USE_OPEN_SET and random.random() < cfg.open_set_mismatch_prob:            # Shuffle text within batch to create mismatches            perm = torch.randperm(ids.size(0))            ids = ids[perm]            attn = attn[perm]            # Zero out masks for mismatched samples (no valid segmentation)            masks = torch.zeros_like(masks)        optimizer.zero_grad(set_to_none=True)        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=cfg.use_bf16):            output = model(imgs, ids, attn)            if cfg.USE_KD and teacher is not None and distill_criterion is not None:                with torch.no_grad():                    teacher_output = teacher(imgs, ids, attn)                    if teacher_output.shape[-2:] != output.shape[-2:]:                        teacher_output = F.interpolate(teacher_output,                                                        size=output.shape[-2:],                                                        mode='bilinear',                                                        align_corners=False)                loss = distill_criterion(output, teacher_output, masks, criterion)            else:                loss = criterion(output, masks)        loss.backward()        optimizer.step()        scheduler.step()        total_loss += loss.item()        avg_loss    = total_loss / (i + 1)        lr          = optimizer.param_groups[0]['lr']        with torch.no_grad():            pred_up  = F.interpolate(output.detach(), size=masks.shape[-2:],                                     mode='bilinear', align_corners=False)            pred_prob = torch.softmax(pred_up, dim=1)[:, 1:, :, :]            metrics.update(pred_prob, masks)        pbar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{avg_loss:.4f}', lr=f'{lr:.2e}')        del imgs, masks, ids, attn, output, loss    train_time = timer.stop(f'Epoch {epoch} train')    epoch_loss = total_loss / N    stats      = metrics.compute()    return epoch_loss, stats

In [ ]:
# ============================================================# 11c. Evaluation# ============================================================@torch.no_grad()def evaluate(model, loader, criterion, epoch,             metrics: SegmentationMetrics, timer: EpochTimer):    model.eval()    metrics.reset()    total_loss = 0.0    thresholds = SegmentationMetrics.THRESHOLDS   # [0.5, 0.7, 0.9]    n_batches = 0    timer.start()    pbar = tqdm(loader, desc=f'Epoch {epoch} [Val]  ', unit='batch',                dynamic_ncols=True, leave=True,                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')    for imgs, masks, ids, attn in pbar:        imgs  = imgs.to(DEVICE, non_blocking=True)        masks = masks.to(DEVICE, non_blocking=True)        ids   = ids.to(DEVICE,  non_blocking=True)        attn  = attn.to(DEVICE, non_blocking=True)        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=cfg.use_bf16):            out  = model(imgs, ids, attn)            loss = criterion(out, masks)        total_loss += loss.item()        n_batches += 1        pred_up   = F.interpolate(out, size=masks.shape[-2:],                                  mode='bilinear', align_corners=False)        pred_prob = torch.softmax(pred_up, dim=1)[:, 1:, :, :]        metrics.update(pred_prob, masks)        pbar.set_postfix(loss=f'{total_loss/n_batches:.4f}')    val_time = timer.stop(f'Epoch {epoch} val')    val_loss = total_loss / max(n_batches, 1)    stats    = metrics.compute()    for th in thresholds:        key = f'Pr@{th}'        print(f'         {key}: {stats[key]*100:.2f}%')    return val_loss, stats

In [ ]:
# ============================================================# 11d. Main training loop (stage-aware)# ============================================================epoch_timer   = EpochTimer()train_metrics = SegmentationMetrics()val_metrics   = SegmentationMetrics()logger        = EpochLogger(save_dir=cfg.output_dir)start_epoch, best_oiou = load_checkpoint(model, optimizer, scheduler, cfg)gc.collect()torch.cuda.empty_cache()patience        = getattr(cfg, 'patience', 5)epochs_no_improve = 0print(f'\n{"="*60}')print(f' Stage     : {cfg.stage_name}')print(f' Training  : epoch {start_epoch} -> {cfg.epochs - 1}')print(f' Device    : {DEVICE} ({torch.cuda.get_device_name(0)})')print(f' Fusion    : {cfg.fusion_type}')print(f' Augment   : {cfg.USE_AUGMENTATION}')print(f' Open-Set  : {cfg.USE_OPEN_SET}')print(f' KD        : {cfg.USE_KD}')print(f' Output    : {cfg.output_dir}')print(f'{"="*60}\n')t0 = time.time()epoch_pbar = tqdm(range(start_epoch, cfg.epochs),                  desc='Overall', unit='epoch',                  dynamic_ncols=True, leave=True,                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} epochs [{elapsed}<{remaining}]')for epoch in epoch_pbar:    train_loss, train_stats = train_one_epoch(        model, train_loader, criterion, optimizer, scheduler, epoch, cfg,        metrics=train_metrics, timer=epoch_timer,        teacher=teacher_model, distill_criterion=distill_criterion,    )    if epoch == start_epoch:        peak = torch.cuda.max_memory_allocated(0) / 1024**3        total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3        print(f'\n  [GPU peak memory: {peak:.2f} / {total_mem:.1f} GB]')        torch.cuda.reset_peak_memory_stats()    val_loss, val_stats = evaluate(        model, val_loader, criterion, epoch,        metrics=val_metrics, timer=epoch_timer    )    is_best = val_stats["oIoU"] * 100 > best_oiou    if is_best:        best_oiou = val_stats["oIoU"] * 100        epochs_no_improve = 0    else:        epochs_no_improve += 1    save_checkpoint(model, optimizer, scheduler, epoch, best_oiou, cfg, is_best)    logger.log(        epoch,        train_loss = train_loss,        val_loss   = val_loss,        train_oIoU = train_stats["oIoU"],        val_oIoU   = val_stats["oIoU"],        train_mIoU = train_stats["mIoU"],        val_mIoU   = val_stats["mIoU"],        **{f"val_{k}": v for k, v in val_stats.items() if k.startswith("Pr@")},    )    print(f'--> Epoch {epoch}/{cfg.epochs-1} | '          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '          f'Val oIoU: {val_stats["oIoU"]*100:.2f}% | '          f'Best: {best_oiou:.2f}% | '          f'Patience: {epochs_no_improve}/{patience}')    epoch_pbar.set_postfix(        val_oIoU=f'{val_stats["oIoU"]*100:.2f}%',        best=f'{best_oiou:.2f}%')    gc.collect()    torch.cuda.empty_cache()    if epochs_no_improve >= patience:        print(f'\n  Early stopping — no improvement for {patience} epochs.')        breaktotal_time = time.time() - t0print(f'\n{"="*60}')print(f' {cfg.stage_name} complete!')print(f' Total time : {datetime.timedelta(seconds=int(total_time))}')print(f' Best oIoU  : {best_oiou:.2f}%')print(f'{"="*60}\n')epoch_timer.summary()# Save results CSV for this stageimport csvcsv_path = os.path.join(cfg.output_dir, f'results_{cfg.stage_name}.csv')keys = list(logger.history.keys())with open(csv_path, 'w', newline='') as f:    writer = csv.DictWriter(f, fieldnames=keys)    writer.writeheader()    for i in range(len(logger.history['epoch'])):        writer.writerow({k: logger.history[k][i] for k in keys})print(f'Results CSV: {csv_path}')logger.save_json(f'training_log_{cfg.stage_name}.json')logger.plot_curves(f'training_curves_{cfg.stage_name}.png')

---## 12. Testing

In [ ]:
# ============================================================# 12. Test Set Evaluation# ============================================================@torch.no_grad()def evaluate_test(model, loader, criterion, cfg, metrics: SegmentationMetrics):    model.eval()    metrics.reset()    total_loss = 0.0    n_batches  = 0    pbar = tqdm(loader, desc='Testing', unit='batch',                dynamic_ncols=True, leave=True)    for imgs, masks, ids, attn in pbar:        imgs  = imgs.to(DEVICE, non_blocking=True)        masks = masks.to(DEVICE, non_blocking=True)        ids   = ids.to(DEVICE,  non_blocking=True)        attn  = attn.to(DEVICE, non_blocking=True)        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=cfg.use_bf16):            out  = model(imgs, ids, attn)            loss = criterion(out, masks)        total_loss += loss.item()        n_batches  += 1        pred_up  = F.interpolate(out, size=masks.shape[-2:],                                 mode='bilinear', align_corners=False)        pred_bin = pred_up.argmax(dim=1, keepdim=True).float()        metrics.update(pred_bin, masks)    test_loss = total_loss / max(n_batches, 1)    stats     = metrics.compute()    return test_loss, stats# Load best checkpointpaths = _stage_paths(cfg)if os.path.exists(paths['best']):    ckpt = torch.load(paths['best'], map_location=DEVICE)    model.load_state_dict(ckpt['model'], strict=False)    print(f'Loaded best {cfg.stage_name} checkpoint (epoch {ckpt["epoch"]})')test_metrics = SegmentationMetrics()test_loss, test_stats = evaluate_test(model, test_loader, criterion, cfg, test_metrics)print(f'\n{"="*60}')print(f'  TEST RESULTS — {cfg.stage_name}')print(f'{"="*60}')rows = [    ['Test Loss',  f'{test_loss:.4f}'],    ['oIoU',       f'{test_stats["oIoU"]*100:.2f}%'],    ['mIoU',       f'{test_stats["mIoU"]*100:.2f}%'],]for k in SegmentationMetrics.THRESHOLDS:    rows.append([f'Pr@{k}', f'{test_stats[f"Pr@{k}"]*100:.2f}%'])print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='rounded_outline'))# Saveresults_path = os.path.join(cfg.output_dir, f'test_results_{cfg.stage_name}.json')with open(results_path, 'w') as f:    json.dump({'stage': cfg.stage_name, 'test_loss': test_loss,               **{k: float(v) for k, v in test_stats.items()}}, f, indent=2)print(f'\nResults: {results_path}')

---## 13. Cross-Stage ComparisonLoad results CSVs from all completed stages and compare.

In [ ]:
# ============================================================# 13. Cross-stage comparison# ============================================================import csvimport globstage_names = ['stage0_baseline', 'stage1_aug', 'stage2_aug_openset', 'stage3_aug_openset_kd']stage_labels = {    'stage0_baseline': ('Stage 0: Baseline', '#2196F3'),    'stage1_aug': ('Stage 1: + Augmentation', '#4CAF50'),    'stage2_aug_openset': ('Stage 2: + Open-Set', '#FF9800'),    'stage3_aug_openset_kd': ('Stage 3: + KD', '#F44336'),}all_stage_data = {}for sname in stage_names:    csv_path = os.path.join(cfg.output_dir, f'results_{sname}.csv')    if os.path.exists(csv_path):        data = defaultdict(list)        with open(csv_path) as f:            reader = csv.DictReader(f)            for row in reader:                for k, v in row.items():                    data[k].append(float(v))        all_stage_data[sname] = data        print(f'  Loaded: {sname} ({len(data["epoch"])} epochs)')if len(all_stage_data) < 2:    print('\nNeed at least 2 stages to compare. Train more stages first.')else:    metrics_to_plot = [        ('val_loss',  'Val Loss'),        ('val_oIoU',  'Val oIoU'),        ('val_mIoU',  'Val mIoU'),    ]    fig, axes = plt.subplots(1, 3, figsize=(18, 5))    fig.suptitle('Cross-Stage Comparison', fontsize=14, fontweight='bold')    for ax, (key, title) in zip(axes.flat, metrics_to_plot):        for sname, data in all_stage_data.items():            label, color = stage_labels.get(sname, (sname, 'gray'))            if key in data:                ax.plot(data['epoch'], data[key], label=label, color=color,                        linewidth=2, marker='o', markersize=3)        ax.set_title(title)        ax.set_xlabel('Epoch')        ax.legend(fontsize=8)        ax.grid(True, alpha=0.3)    plt.tight_layout()    plt.savefig(os.path.join(cfg.output_dir, 'cross_stage_comparison.png'),                dpi=150, bbox_inches='tight')    plt.show()    # Summary table    print('\n' + '=' * 80)    print('  CROSS-STAGE BEST-EPOCH SUMMARY')    print('=' * 80)    summary = []    baseline_oiou = None    for sname in stage_names:        if sname not in all_stage_data:            continue        d = all_stage_data[sname]        best_idx = int(np.argmax(d['val_oIoU']))        oiou = d['val_oIoU'][best_idx] * 100        miou = d['val_mIoU'][best_idx] * 100        vloss = d['val_loss'][best_idx]        if sname == 'stage0_baseline':            baseline_oiou = oiou        delta = f'{oiou - baseline_oiou:+.2f}pp' if baseline_oiou is not None else '-'        label = stage_labels.get(sname, (sname, ''))[0]        summary.append([label, int(d['epoch'][best_idx]), f'{vloss:.4f}',                        f'{oiou:.2f}%', f'{miou:.2f}%', delta])    print(tabulate(summary,                   headers=['Stage', 'Best Ep', 'Val Loss', 'oIoU', 'mIoU', 'Δ vs Base'],                   tablefmt='rounded_outline'))

---## Next Steps1. **Stage 0**: Set all toggles to `False`, run training → baseline results2. **Stage 1**: Set `USE_AUGMENTATION = True`, run → augmentation comparison3. **Stage 2**: Set `USE_AUGMENTATION = True, USE_OPEN_SET = True`, run → open-set results4. **Stage 3**: Set all three to `True`, run → full pipeline with KDEach stage creates its own checkpoint and results CSV. Section 13 compares all stages.